# 1. Instalar librerias

In [1]:
!pip install flask
!pip install pyngrok

!pip install supervision
!pip install ultralytics
!pip install trackers
!pip install tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.2/107.2 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 460.4/460.4 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.0/204.0 kB 21.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.7/224.7 kB 17.2 MB/s eta 0:00:00


# 2. Importar librerias

In [2]:
from supervision.annotators.utils import ColorLookup
from supervision.draw.color import ColorPalette
from google.colab.patches import cv2_imshow
from IPython.display import clear_output
from trackers import ByteTrackTracker
from ultralytics import YOLO
from typing import List
from tqdm import tqdm

import supervision as sv
import numpy as np
import cv2
import os

HOME = os.getcwd()
print(HOME)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
/content


In [3]:
%cd {HOME}
!wget --load-cookies /tmp/cookies.txt "https://docs.google.com/uc?export=download&confirm=$(wget --quiet --save-cookies /tmp/cookies.txt --keep-session-cookies --no-check-certificate 'https://docs.google.com/uc?export=download&id=1pz68D1Gsx80MoPg-_q-IbEdESEmyVLm-' -O- | sed -rn 's/.*confirm=([0-9A-Za-z_]+).*/\1\n/p')&id=1pz68D1Gsx80MoPg-_q-IbEdESEmyVLm-" -O vehicle-counting.mp4 && rm -rf /tmp/cookies.txt

/content
--2026-10-17 01:08:57--  https://docs.google.com/uc?export=download&confirm=&id=1pz68D1Gsx80MoPg-_q-IbEdESEmyVLm-
Resolving docs.google.com (docs.google.com)... 142.250.4.100, 142.250.4.138, 142.250.4.101, ...
Connecting to docs.google.com (docs.google.com)|142.250.4.100|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1pz68D1Gsx80MoPg-_q-IbEdESEmyVLm-&export=download [following]
--2026-10-17 01:08:58--  https://drive.usercontent.google.com/download?id=1pz68D1Gsx80MoPg-_q-IbEdESEmyVLm-&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 142.251.12.132, 2404:6800:4003:c11::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|142.251.12.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 35345757 (34M) [video/mp4]
Saving to: ‘vehicle-counting.mp4’

vehicle-counting.mp 100%[===================>]  33.71M  22.3MB/s

# 3. Procesar frame

In [4]:
class VideoProcessor:
  def __init__(
      self,
      source_weights_path: str,
      source_video_path: str,
      target_video_path: str,
      confidence_threshold: float = 0.3,
      iou_threshold: float = 0.7,
  ) -> None:
    self.source_video_path = source_video_path
    self.target_video_path = target_video_path
    self.conf_threshold = confidence_threshold
    self.iou_threshold = iou_threshold

    self.model = YOLO(source_weights_path)
    self.model.fuse()

    self.CLASS_NAMES_DICT = self.model.model.names
    self.CLASS_ID = [2, 3, 5, 7]

    self.LINE_START = sv.Point(50, 1500)
    self.LINE_END = sv.Point(3840-50, 1500)

    # create BYTETracker instance
    self.tracker = ByteTrackTracker()
    # create VideoInfo instance
    self.video_info = sv.VideoInfo.from_video_path(self.source_video_path)
    # create LineCounter instance
    self.line_counter = sv.LineZone(start=self.LINE_START, end=self.LINE_END)
    # create instance of BoundingBoxAnnotator
    self.boundingBoxAnnotator = sv.BoxAnnotator(color=ColorPalette.DEFAULT, thickness=2, color_lookup=ColorLookup.CLASS)
    # create instance of LineZoneAnnotator
    self.line_annotator = sv.LineZoneAnnotator(thickness=4, text_thickness=4, text_scale=2)
    # creare instance of LabelAnnotator
    self.label_annotator = sv.LabelAnnotator()


  def process_video(self):
    generator = sv.get_video_frames_generator(self.source_video_path)

    with sv.VideoSink(self.target_video_path, self.video_info) as sink:
        for frame in tqdm(generator, total=self.video_info.total_frames):
            annotated_frame = self.process_frame(frame)
            sink.write_frame(annotated_frame)

  def annotate_frame(
      self, frame: np.ndarray, detections: sv.Detections
  ) -> np.ndarray:
    annotated_frame = frame.copy()

    # format custom labels
    labels = [f"#{tracker_id}" for tracker_id in detections.tracker_id]

    # updating line counter
    self.line_counter.trigger(detections=detections)

    # annotate and display frame
    annotated_frame = self.boundingBoxAnnotator.annotate(annotated_frame, detections)
    annotated_frame = self.line_annotator.annotate(annotated_frame, self.line_counter)
    annotated_frame = self.label_annotator.annotate(annotated_frame, detections, labels=labels)

    return annotated_frame

  def process_frame(self, frame: np.ndarray) -> np.ndarray:
    # model prediction on single frame and conversion to supervision Detections
    results = self.model(frame, verbose=False, conf=self.conf_threshold, iou=self.iou_threshold)[0]
    detections = sv.Detections.from_ultralytics(results)

    # filtering out detections with unwanted classes
    mask = np.array([class_id in self.CLASS_ID for class_id in detections.class_id], dtype=bool)
    detections = detections[mask]

    # filtering out detections by tracking
    detections = self.tracker.update(detections)
    return self.annotate_frame(frame, detections)

In [6]:
processor = VideoProcessor(
      source_weights_path="yolov8x.pt",
      source_video_path=f"{HOME}/vehicle-counting.mp4",
      target_video_path=f"{HOME}/vehicle-counting-output.mp4",
      confidence_threshold=0.3,
      iou_threshold=0.5,
  )
processor.process_video()

YOLOv8x summary (fused): 112 layers, 68,200,608 parameters, 0 gradients, 257.8 GFLOPs


100%|██████████| 538/538 [01:32<00:00,  5.84it/s]


In [ ]:
#29XvlxdOkJ3n0sb50lq6TfwNYiL_6gPTtXhVmCyCgzDwD6LmD

import getpass
import os
import threading

from flask import Flask
from pyngrok import ngrok, conf

print("Enter your authtoken, which can be copied from https://dashboard.ngrok.com/get-started/your-authtoken")
conf.get_default().auth_token = getpass.getpass()

app = Flask(__name__)
port = "8000"

# Open a ngrok tunnel to the HTTP server
public_url = ngrok.connect(port).public_url
print(f" * ngrok tunnel \"{public_url}\" -> \"http://127.0.0.1:{port}\"")

# Update any base URLs to use the public ngrok URL
app.config["BASE_URL"] = public_url

# ... Update inbound traffic via APIs to use the public-facing ngrok URL


# Define Flask routes
@app.route("/")
def index():
    return "Hello from Colab!"

# Start the Flask server in a new thread
threading.Thread(target=app.run, kwargs={"use_reloader": False}).start()